# 01 - Data Overview

Dataset summary, schema inference, missing values, and target distribution.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import plotly.express as px
from src.data.data_loader import infer_schema, load_raw_data
from src.pipelines.validator import DataValidator

In [2]:
df = load_raw_data()
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head()

2026-06-10 08:47:33 | INFO | src.ingestion.data_loader | Loading raw data from C:\Users\JOY\Downloads\Telco Churn\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv
2026-06-10 08:47:33 | INFO | src.ingestion.data_loader | Loaded 7043 rows and 21 columns
Rows: 7,043 | Columns: 21


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
schema = infer_schema(df)
schema_df = pd.DataFrame(
    [
        {
            "column": col.name,
            "dtype": col.dtype,
            "semantic_type": col.semantic_type,
            "null_count": col.null_count,
            "unique_count": col.unique_count,
        }
        for col in schema.columns
    ]
)
schema_df

,column,dtype,semantic_type,null_count,unique_count
0,customerID,object,id,0,7043
1,gender,object,binary,0,2
2,SeniorCitizen,int64,numeric,0,2
3,Partner,object,binary,0,2
4,Dependents,object,binary,0,2
5,tenure,int64,numeric,0,73
6,PhoneService,object,binary,0,2
7,MultipleLines,object,categorical,0,3
8,InternetService,object,categorical,0,3
9,OnlineSecurity,object,categorical,0,3


In [4]:
df.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customerID,7043,7043,7590-VHVEG,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,7043,2,Male,3555,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,7043.0,NaN,NaN,NaN,0.162147,0.368612,0.0,0.0,0.0,0.0,1.0
Partner,7043,2,No,3641,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,7043,2,No,4933,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,7043.0,NaN,NaN,NaN,32.371149,24.559481,0.0,9.0,29.0,55.0,72.0
PhoneService,7043,2,Yes,6361,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,7043,3,No,3390,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,7043,3,Fiber optic,3096,NaN,NaN,NaN,NaN,NaN,NaN,NaN
OnlineSecurity,7043,3,No,3498,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
validator = DataValidator()
report = validator.validate(df)
print("Missing values:", report["missing_values"])
print("Duplicates:", report["duplicates"])
print("Target distribution:", report["target_distribution"])

2026-06-10 08:47:33 | INFO | src.preprocessing.validator | Validation completed with 0 issues
Missing values: {'TotalCharges': 11}
Duplicates: {'duplicate_ids': 0, 'duplicate_rows': 0}
Target distribution: {'No': 5174, 'Yes': 1869}


In [6]:
churn_counts = df["Churn"].value_counts().reset_index()
churn_counts.columns = ["Churn", "Count"]
fig = px.pie(churn_counts, names="Churn", values="Count", title="Churn Distribution")
fig.show()